# GEANT h96 partial DG-STMTL comparison — reporting only

This workflow validates the pinned source package, reads only 11 explicitly named run.json manifests from the approved Drive backups, writes a fresh validation report, and copies it back with SHA-256 read-back verification. It does not load checkpoints, predictions, arrays, or test artifacts; it does not train or run inference. The comparison is partial because original GNN Topology seed 42 is absent.

In [ ]:
from pathlib import Path
import hashlib, io, json, os, shutil, stat, subprocess, sys, uuid, zipfile
from google.colab import files

EXPECTED_SOURCE_ZIP_SHA256 = "1718b4fffbf4d128283e15c6dfd362ea07fbbd9726fed2441b5f2ff3ca618682"
EXPECTED_SOURCE_MANIFEST_SHA256 = "4825750b29d8a50395ed3af289817df0310c9f09b52c8f6a1b5ed70dd791285f"
EXPECTED_BUNDLE_ID = "geant-h96-partial-dg-stmtl-report-v2"
EXPECTED_MEMBERS = {"README.md", "phase9_geant/compare_dg_stmtl_h96.py"}
SESSION_ID = uuid.uuid4().hex
STAGE = Path('/content') / f'geant_h96_partial_dg_report_v2_{SESSION_ID}'
if STAGE.exists() or STAGE.is_symlink(): raise FileExistsError(f'Refusing existing staging path: {STAGE}')
def sha256_bytes(value): return hashlib.sha256(value).hexdigest()

print('Upload exactly the reporting source ZIP and source_snapshot_manifest.json; names may have (1) suffixes, hashes identify them.')
uploads = files.upload()
wanted = {EXPECTED_SOURCE_ZIP_SHA256: 'source_zip', EXPECTED_SOURCE_MANIFEST_SHA256: 'source_manifest'}
matched = {}
for uploaded_name, uploaded_bytes in uploads.items():
    digest = sha256_bytes(uploaded_bytes)
    if digest not in wanted: raise RuntimeError(f'FAIL unrecognized upload: {uploaded_name} ({digest})')
    role = wanted[digest]
    if role in matched: raise RuntimeError(f'FAIL duplicate upload for {role}')
    matched[role] = uploaded_bytes
if set(matched) != {'source_zip', 'source_manifest'}: raise RuntimeError('FAIL both pinned files are required')
if sha256_bytes(matched['source_zip']) != EXPECTED_SOURCE_ZIP_SHA256: raise RuntimeError('FAIL source ZIP hash')
if sha256_bytes(matched['source_manifest']) != EXPECTED_SOURCE_MANIFEST_SHA256: raise RuntimeError('FAIL source manifest hash')
source_manifest = json.loads(matched['source_manifest'].decode('utf-8'))
if source_manifest.get('bundle_id') != EXPECTED_BUNDLE_ID or source_manifest.get('archive_sha256') != EXPECTED_SOURCE_ZIP_SHA256: raise RuntimeError('FAIL bundle identity or archive pin')
member_rows = source_manifest.get('members')
if not isinstance(member_rows, list) or {item.get('path') for item in member_rows} != EXPECTED_MEMBERS or len(member_rows) != len(EXPECTED_MEMBERS): raise RuntimeError('FAIL manifest member allowlist')
STAGE.mkdir(parents=False, exist_ok=False)
EXTRACTED = STAGE / 'approved_reporting_source'
EXTRACTED.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(matched['source_zip']), 'r') as archive:
    infos = archive.infolist()
    names = [item.filename for item in infos]
    if len(names) != len(set(names)) or set(names) != EXPECTED_MEMBERS: raise RuntimeError('FAIL archive inventory or duplicate names')
    rows = {item['path']: item for item in member_rows}
    for item in infos:
        name = item.filename.replace('\\', '/')
        mode = (item.external_attr >> 16) & 0xFFFF
        if name.startswith('/') or any(part in ('', '.', '..') for part in name.split('/')): raise RuntimeError(f'FAIL unsafe member path: {name}')
        if stat.S_ISLNK(mode) or item.is_dir(): raise RuntimeError(f'FAIL unsupported ZIP member: {name}')
        payload = archive.read(item)
        expected = rows[name]
        if len(payload) != expected.get('bytes') or sha256_bytes(payload) != expected.get('sha256'): raise RuntimeError(f'FAIL member hash/size: {name}')
        destination = EXTRACTED / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() or destination.is_symlink(): raise FileExistsError(f'Refusing extracted path: {destination}')
        with destination.open('xb') as stream: stream.write(payload)
COMPARE = EXTRACTED / 'phase9_geant' / 'compare_dg_stmtl_h96.py'
if sha256_bytes(COMPARE.read_bytes()) != rows['phase9_geant/compare_dg_stmtl_h96.py']['sha256']: raise RuntimeError('FAIL extracted comparator identity')
help_run = subprocess.run([sys.executable, str(COMPARE), '--help'], text=True, capture_output=True, check=False)
if help_run.returncode != 0 or '--allow-incomplete-gnn-matrix' not in help_run.stdout: raise RuntimeError(f'FAIL comparator CLI check: {help_run.stderr}')
print({'stage': 'package_verification', 'status': 'PASS', 'bundle_id': EXPECTED_BUNDLE_ID, 'training': False, 'test_access': False})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/SDNResearch').resolve(strict=True)
if not DRIVE_BASE.is_dir(): raise RuntimeError('FAIL approved SDNResearch Drive root unavailable')
MATRIX3_RUNS = DRIVE_BASE / 'geant_h96_matrix3_5fe371f8aab74f42983569579477c328' / 'runs'
DG_RUNS = DRIVE_BASE / 'geant_dg_stmtl_h96_v1_review3' / 'runs'
review18_text = input('Enter exact existing Review18 recovery backup ROOT (folder containing runs/): ').strip()
review18_entered = Path(os.path.abspath(review18_text))
if review18_entered.is_symlink(): raise RuntimeError('FAIL Review18 root is a symlink')
REVIEW18_ROOT = review18_entered.resolve(strict=True)
if REVIEW18_ROOT.parent != DRIVE_BASE or not REVIEW18_ROOT.name.startswith('geant_h96_matrix3_recovery_review18_'): raise RuntimeError('FAIL Review18 root is outside approved parent or has unexpected name')
REVIEW18_RUNS = REVIEW18_ROOT / 'runs'
for folder in (MATRIX3_RUNS, DG_RUNS, REVIEW18_RUNS):
    if not folder.is_dir() or folder.is_symlink() or folder.resolve(strict=True) != folder: raise RuntimeError(f'FAIL missing or redirected backup folder: {folder}')
def manifest_path(root, run_name):
    run_dir = root / run_name
    path = run_dir / 'run.json'
    if run_dir.is_symlink() or path.is_symlink() or not path.is_file() or path.resolve(strict=True) != path: raise RuntimeError(f'FAIL missing or redirected manifest: {path}')
    return path
gnn_manifest_paths = [
    manifest_path(MATRIX3_RUNS, 'h96_topology_seed_123'), manifest_path(MATRIX3_RUNS, 'h96_topology_seed_2026'),
    manifest_path(MATRIX3_RUNS, 'h96_identity_seed_42'), manifest_path(MATRIX3_RUNS, 'h96_identity_seed_123'), manifest_path(MATRIX3_RUNS, 'h96_identity_seed_2026'),
    manifest_path(MATRIX3_RUNS, 'h96_shuffled_seed_42'),
    manifest_path(REVIEW18_RUNS, 'h96_shuffled_seed_123'), manifest_path(REVIEW18_RUNS, 'h96_shuffled_seed_2026'),
]
dg_manifest_paths = [manifest_path(DG_RUNS, f'dg_stmtl_topology_seed_{seed}') for seed in (42, 123, 2026)]
if len(set(gnn_manifest_paths)) != 8 or len(set(dg_manifest_paths)) != 3: raise RuntimeError('FAIL duplicate manifest paths')
print({'stage': 'manifest_allowlist', 'status': 'PASS', 'gnn_manifests': 8, 'dg_manifests': 3, 'missing_original_gnn': {'adjacency_mode': 'topology', 'seed': 42}, 'opened_kind': 'run.json only'})

In [ ]:
comparison_dir = STAGE / 'comparison'
cli = [sys.executable, str(COMPARE)]
for path in gnn_manifest_paths: cli.extend(['--gnn-run-json', str(path)])
for path in dg_manifest_paths: cli.extend(['--dg-run-json', str(path)])
cli.extend(['--allow-incomplete-gnn-matrix', '--output-directory', str(comparison_dir)])
run = subprocess.run(cli, text=True, capture_output=True, check=False)
print(run.stdout.strip())
if run.returncode != 0: raise RuntimeError(f'FAIL comparison exited {run.returncode}:\n{run.stderr}')
result = json.loads((comparison_dir / 'comparison.json').read_text(encoding='utf-8'))
if result.get('status') != 'partial' or result.get('missing_gnn_combinations') != [{'model': 'original_gnn', 'adjacency_mode': 'topology', 'seed': 42}]: raise RuntimeError('FAIL approved missing combination not recorded')
if result.get('ewma_consistency_check') != {'absolute_tolerance': 1e-9, 'relative_tolerance': 0.0, 'policy': 'compare separately recorded EWMA values without rewriting them; fixed absolute tolerance only'}: raise RuntimeError('FAIL EWMA consistency policy missing')
if result.get('test_access') is not False or result.get('aggregate', {}).get('gnn_topology', {}).get('n') != 2: raise RuntimeError('FAIL comparison scope or test-access declaration')
if any(result['aggregate'][name]['n'] != 3 for name in ('gnn_identity', 'gnn_shuffled', 'dg_stmtl_based')): raise RuntimeError('FAIL method sample counts')
print('Compact validation comparison:')
for method, values in result['aggregate'].items(): print(f"{method}: n={values['n']}, MAE={values['mae']['mean']:.8g}, RMSE={values['rmse']['mean']:.8g}")
print({'stage': 'comparison', 'status': 'PASS', 'scope': result['comparison_scope'], 'training': False, 'inference': False, 'test_access': False})

drive_output = DRIVE_BASE / f'geant_h96_partial_dg_stmtl_comparison_v2_{SESSION_ID}'
if drive_output.exists() or drive_output.is_symlink(): raise FileExistsError(f'Refusing existing report directory: {drive_output}')
if drive_output.parent != DRIVE_BASE or DRIVE_BASE.is_symlink(): raise RuntimeError('FAIL Drive report path containment')
drive_output.mkdir(parents=False, exist_ok=False)
report_hashes = {}
for name in ('comparison.json', 'comparison.csv', 'report.md'):
    src, dst = comparison_dir / name, drive_output / name
    if not src.is_file() or dst.exists() or dst.is_symlink(): raise RuntimeError(f'FAIL report copy path: {name}')
    digest = sha256_bytes(src.read_bytes())
    with src.open('rb') as source, dst.open('xb') as target: shutil.copyfileobj(source, target)
    if sha256_bytes(dst.read_bytes()) != digest: raise RuntimeError(f'FAIL report read-back hash: {name}')
    report_hashes[name] = {'sha256': digest, 'readback_sha256': sha256_bytes(dst.read_bytes()), 'bytes': dst.stat().st_size}
copy_record = {'status': 'copied_and_readback_verified', 'comparison_status': 'partial', 'missing_original_gnn': {'adjacency_mode': 'topology', 'seed': 42}, 'ewma_consistency_check': result['ewma_consistency_check'], 'files': report_hashes, 'training': False, 'test_access': False}
with (drive_output / 'copy_verification.json').open('xb') as stream: stream.write((json.dumps(copy_record, sort_keys=True, indent=2) + '\n').encode('utf-8'))
if sha256_bytes((drive_output / 'copy_verification.json').read_bytes()) != sha256_bytes((json.dumps(copy_record, sort_keys=True, indent=2) + '\n').encode('utf-8')): raise RuntimeError('FAIL copy record read-back hash')
print({'stage': 'Drive_report_copy', 'status': 'PASS', 'directory': str(drive_output), 'files': report_hashes, 'reporting_only': True})